# Experimental Validation Pipeline

This notebook demonstrates how to load recorded audio from a hardware prototype and compare it to the simulation implemented in this repository. Recordings can be made with a cheap LED and modulation source, a microphone placed near a ballast or transformer, or a photoacoustic cell.

Replace the `recording_path` variable in the code cell below with the path to your recorded WAV file. The notebook will load the recording using `soundfile`, synthesise an artificial signal using the same parameters, overlay the two signals in the time domain and compute a spectrogram difference plot. You can modify the parameters to match your experimental setup or run parameter sweeps to study sensitivity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path

# Import the Python models from this repository
import sys
sys.path.insert(0, str(Path('python').resolve()))
import models

# TODO: replace with your recorded WAV file
recording_path = Path('data/recordings/your_recording.wav')
data, fs = sf.read(recording_path)

# Choose a preset matching your experiment or customise parameters
params = models.PRESETS[0].clone()
# Synthesize simulated signal with the same duration and sample rate
sim = models.synthesize(params, len(data) / fs, sample_rate=fs)

# Optionally align signals (e.g. via cross‑correlation) before comparison
# Here we simply truncate to the same length
length = min(len(data), len(sim))
data = data[:length]
sim = sim[:length]

# Plot time domain overlay
plt.figure(figsize=(10, 4))
plt.plot(data[:10000], label='Recording', alpha=0.7)
plt.plot(sim[:10000], label='Simulation', alpha=0.7)
plt.legend()
plt.title('Time domain overlay')
plt.xlabel('Sample index')
plt.ylabel('Amplitude')
plt.show()

# Compute and plot spectrogram difference
from scipy.signal import spectrogram
f_r, t_r, Sxx_r = spectrogram(data, fs)
f_s, t_s, Sxx_s = spectrogram(sim, fs)
# Ensure the matrices have the same shape by trimming
min_time_bins = min(Sxx_r.shape[1], Sxx_s.shape[1])
min_freq_bins = min(Sxx_r.shape[0], Sxx_s.shape[0])
Sxx_r = Sxx_r[:min_freq_bins, :min_time_bins]
Sxx_s = Sxx_s[:min_freq_bins, :min_time_bins]
diff = 20 * np.log10(Sxx_r + 1e-12) - 20 * np.log10(Sxx_s + 1e-12)
plt.figure(figsize=(8, 4))
plt.pcolormesh(t_r[:min_time_bins], f_r[:min_freq_bins], diff, shading='auto')
plt.colorbar(label='Magnitude difference (dB)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram difference')
plt.show()